#### Imports

In [ ]:
import pandas as pd
import numpy as np
import time
import pyodbc
import pickle
import warnings
import os
import cv2
from tensorflow.keras.models import load_model
from pathlib import Path
import winsound

warnings.filterwarnings('ignore')   
print("--- INITIALIZING FACTORY INFERENCE ENGINE ---")

--- INITIALIZING FACTORY INFERENCE ENGINE ---


#### Data Leakage Prevention

In [2]:
def get_image_hash(image):
    # Compute a hash to identify duplicate images regardless of filename
    return hash(image.tobytes())

def remove_cross_split_duplicates(train_path, val_path):
    print("Checking for cross-split duplicates (Data Leakage Prevention)...")
    
    train_hashes = set()
    # Path to your class folders
    classes = ["Violence", "NonViolence"]
    
    # 1. Build a registry of all training image hashes
    for cls in classes:
        cls_dir = Path(train_path) / cls
        if cls_dir.exists():
            for img_path in cls_dir.glob("*.jpg"):
                img = cv2.imread(str(img_path))
                if img is not None:
                    train_hashes.add(get_image_hash(img))

    # 2. Identify and remove any validation images that exist in the training set
    removed = 0
    for cls in classes:
        cls_dir = Path(val_path) / cls
        if cls_dir.exists():
            for img_path in cls_dir.glob("*.jpg"):
                img = cv2.imread(str(img_path))
                if img is not None:
                    if get_image_hash(img) in train_hashes:
                        os.remove(img_path)
                        removed += 1
    
    print(f"Removed from validation: {removed} duplicates.")

#### Define the Target Serial

In [3]:
SELECTED_MACHINE = input("Enter the Machine Serial Number to analyze (e.g., 0521760): ").strip()
print(f"Target Machine Set To: {SELECTED_MACHINE}")

Target Machine Set To: 0521760


#### Azure SQL & Path Configuration

In [4]:
# AZURE SQL CONFIGURATION
AZURE_SERVER = 'kreseakreiotprdsrv.database.windows.net'
AZURE_DATABASE = 'KRESEAKREIOTPRD'
AZURE_USERNAME = 'IOTAdmin'
AZURE_PASSWORD = 'oKuvodump5JNG7dM' 
AZURE_TABLE_NAME = 'MBP_ControllerData' 

# PATHS TO TRAINED FILES
MODEL_PATH = r"D:\Courses\IIT\AIDS\FYP\Preventing-Mechanism\sewing_machine_predictive_lstm.h5"
SCALER_PATH = r'D:\Courses\IIT\AIDS\FYP\Preventing-Mechanism\sewing_scaler.pkl'
ENCODER_PATH = r'D:\Courses\IIT\AIDS\FYP\Preventing-Mechanism\sewing_encoder.pkl'

driver = '{ODBC Driver 17 for SQL Server}'
conn_str = f'DRIVER={driver};SERVER={AZURE_SERVER};PORT=1433;DATABASE={AZURE_DATABASE};UID={AZURE_USERNAME};PWD={AZURE_PASSWORD}'

#### Load Baseline Model and Translators

In [5]:
try:
    model = load_model(MODEL_PATH)
    with open(SCALER_PATH, 'rb') as f:
        scaler = pickle.load(f)
    with open(ENCODER_PATH, 'rb') as f:
        encoder = pickle.load(f)
    print("AI Model and Translators Loaded Successfully.")
except Exception as e:
    print(f"ERROR LOADING FILES: {e}")

AI Model and Translators Loaded Successfully.


#### Logic Engines

In [6]:
maintenance_solutions = {
    "Needle Breakage": "SOLUTION: Immediately replace the needle and check the needle guard alignment.",
    "Thread Jam": "SOLUTION: Clear the bobbin case, check thread tension, and clean the feed dogs.",
    "Blade Blunt": "SOLUTION: Schedule a replacement of the upper and lower cutting blades within 1 hour.",
    "High Foot Pressure": "SOLUTION: Reduce presser foot pressure dial by 2 turns.",
    "Healthy Baseline": "STATUS: Machine operating normally. No action required."
}

def parse_vibration_to_bands(vib_str):
    if pd.isna(vib_str): return {}
    parts = str(vib_str).split(',')
    res = {}
    try:
        for i in range(0, len(parts)-1, 2):
            f_start = int(parts[i])
            f_end = f_start + 10
            res[f"{f_start}-{f_end}Hz"] = int(parts[i+1])
    except Exception:
        pass
    return res

electrical_cols = [ 
    'machineVoltageMean', 'machineVoltageMin', 'machineVoltageMax',
    'machineCurrentMean', 'machineCurrentMin', 'machineCurrentMax',
    'machinePowerMean', 'machinePowerMin', 'machinePowerMax'
]
vibration_bands = [f"{i}-{i+10}Hz" for i in range(10, 610, 10)]
ALL_69_FEATURES = electrical_cols + vibration_bands

#### The Live Monitoring Loop

In [ ]:
last_processed_time = None

try:
    print(f"🔄 Connecting to Azure SQL Database: {AZURE_DATABASE}")
    with pyodbc.connect(conn_str) as conn:
        print(f"✅ Connection Verified. Monitoring Machine: {SELECTED_MACHINE}")
        
        query = f"""
            SELECT TOP 5 * FROM {AZURE_TABLE_NAME} 
            WHERE machineSerialNumber = '{SELECTED_MACHINE}' 
            ORDER BY dateTime DESC
        """
        
        while True:
            df_live = pd.read_sql(query, conn)
            
            if not df_live.empty and len(df_live) >= 5:
                # 1. Timestamps
                db_raw_timestamp = df_live['dateTime'].iloc[0]
                last_record_utc = pd.to_datetime(db_raw_timestamp)
                sl_record_time = last_record_utc + pd.Timedelta(hours=5, minutes=30)
                current_local_time = pd.Timestamp.now()
                
                # 2. Only process fresh records
                if sl_record_time != last_processed_time:
                    last_processed_time = sl_record_time
                    
                    # 3. Pre-processing (LSTM Window)
                    df_live_sorted = df_live.iloc[::-1].reset_index(drop=True)
                    vibs = pd.DataFrame(df_live_sorted['machineVibration'].apply(parse_vibration_to_bands).tolist()).fillna(0)
                    extras = df_live_sorted[electrical_cols]
                    df_window = pd.concat([extras, vibs], axis=1).fillna(0)
                    
                    for col in ALL_69_FEATURES:
                        if col not in df_window.columns:
                            df_window[col] = 0
                    df_window = df_window[ALL_69_FEATURES] 
                    
                    # 4. Inference
                    X_live_scaled = scaler.transform(df_window.values)
                    X_inference = np.array([X_live_scaled])
                    prediction_probs = model.predict(X_inference, verbose=0)
                    predicted_state = encoder.inverse_transform([np.argmax(prediction_probs)])[0]
                    
                    # 5. Output Display
                    print("\n")
                    print(f"--- [MACHINE: {SELECTED_MACHINE}] ---")
                    print(f"AZURE RECORD (UTC) : {str(db_raw_timestamp).split('.')[0]}") 
                    print(f"RECORD TIME (SL)   : {sl_record_time.strftime('%Y-%m-%d %H:%M:%S')}")
                    print(f"CURRENT LOCAL TIME : {current_local_time.strftime('%Y-%m-%d %H:%M:%S')}")
                    print(f"PREDICTED STATE    : {predicted_state}")
                    
                    if predicted_state != "Healthy Baseline":
                        print(f"ALERT: {predicted_state} DETECTED")
                        winsound.Beep(1000, 500)
                        if predicted_state in maintenance_solutions:
                            print(f"ACTION: {maintenance_solutions[predicted_state]}")
                    else:
                        print("STATUS: Machine health is optimal")
                    print("-" * 30)
                
            elif df_live.empty:
                print(f"No records found for Serial {SELECTED_MACHINE}.", end="\r")
            
            time.sleep(1)

# NEW: Specific exception for when the user stops the script manually
except KeyboardInterrupt:
    print("\n🛑 Monitoring stopped by user. Database connection closed.")

except Exception as e:
    print(f"\n❌ SCRIPT ERROR: {e}")

🔄 Connecting to Azure SQL Database: KRESEAKREIOTPRD
✅ Connection Verified. Monitoring Machine: 0521760


--- [MACHINE: 0521760] ---
AZURE RECORD (UTC) : 2026-03-07 09:46:09
RECORD TIME (SL)   : 2026-03-07 15:16:09
CURRENT LOCAL TIME : 2026-03-08 23:25:05
PREDICTED STATE    : Healthy Baseline
STATUS: Machine health is optimal
------------------------------

🛑 Monitoring stopped by user. Database connection closed.
